In [14]:
import json
import ollama
import requests
import os
import faiss
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer 

In [15]:

# CONFIGURATION
PDF_PATH = os.environ.get("PDF_PATH", "Data/sample_rag_poc_handbook.pdf")
DB_PATH = "faiss_db"
INDEX_PATH = os.environ.get("INDEX_PATH", f"{DB_PATH}/faiss_index.faiss")
CHUNK_PATH = os.environ.get("CHUNKS_PATH", f"{DB_PATH}/chunks.json")
EMBEDDING_MODEL_NAME = os.environ.get("EMBEDDING_MODEL_NAME", "all-MiniLM-L6-v2")
OLLAMA_MODEL = "llama3.2:3b"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
TOP_K = 4
OLLAMA_URL = "http://localhost:11434/api/chat"

# LOAD EMEDDING MODEL
print("Loading embedding model.....")
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

Loading embedding model.....


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1639.56it/s]


In [16]:
# EXTRACT PDF TEXT


def extract_pdf_text(pdf_path):
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"PDF not found:{pdf_path}")
    reader = PdfReader(pdf_path)
    pages = []

    for page_number, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        if text.strip():
            pages.append({"pages": page_number + 1, "text": text})
            if not pages:
                raise ValueError("No extractable text found in PDF.")
            return pages

In [17]:
def create_chunk(pages):
    chunks = []
    for page in pages:
        words = page["text"].split()
        start = 0
        while start < len(words):
            end = start + CHUNK_SIZE
            chunk_words = words[start:end]
            chunk_text = " ".join(chunk_words)
            if chunk_text.strip():
                chunks.append({"text": chunk_text, "page": page["pages"]})
            if end >= len(words):
                break
            start = end - CHUNK_OVERLAP

    return chunks


In [18]:
# CREATE EMBEDDING


def create_embeddings(chunks):
    texts = [chunks["text"] for chunk in chunks]
    embedding = embedding_model.encode(
        texts, convert_to_numpy=True, normalize_embeddings=True
    )
    return embedding.astype("float32")

In [19]:
# STORE IN FAISS

def save_faiss_index(embeddings,chunks):
    os.makedirs(DB_PATH,exist_ok=True)
    dimension=embeddings.shape[1]
    index=faiss.IndexFlatIp(dimension)
    index.add(embeddings)
    faiss.write_index(index,INDEX_PATH)
    with open(CHUNKS_PATH,"w",encoding="utf-8") as file:
        json.dump(chunks,file,ensure_ascii=False,indent=4)
        print("FAISS index saved successfully.")




In [20]:
os.makedirs("faiss_db",exist_ok=True)
# os.makedirs(os.path.dirname(INDEX_PATH),exist_ok=True)

In [21]:
# LOAD FAISS DATA

def load_faiss_data():
    if not os.path.exists(INDEX_PATH):
        raise FileNotFoundError("FAISS index not found.Build the index first.")
        if not os.path.exists(CHUNKS_PATH):
         raise FileNotFoundError("Chunks file not found.")
        index=faiss.read_index(INDEX_PATH)
        with open(CHUNK_PATH,"r",encoding="utf-8") as file:
            chunks=json.load(file)
            if index.ntotal!=len(chunks):
                raise ValueError("FAISS index and chunks count do not match.")
                return index,chunks
        

In [22]:
# RETRIEVE RELEVANT CHUNKS

def retrieve_chunks(question,index,chunks):
    query_embedding=embedding_model.encode([question],convert_to_numpy=True,normalize_embeddings=True).astype("float32")
    scores,indices=index.search(query_embedding,min(TOP_K,index.ntotal))
    results=[]
    for score,idx in zip(scores[0],indices[0]):
        if idx==-1:
            continue
        result=chunks[idx].copy()
        result["score"]=float(score)
        results.append(result)
        return results

In [23]:
# GENERATING ANSWER USING OLLAMA


def generate_answer(question, retrieved_chunks, model="OLLAMA_MODEL"):
    # Combine your retrieved Faiss chunks into a single context string
    context = "\n\n".join(
        f"[Page {chunk['page']}]: {chunk['text']}" for chunk in retrieved_chunks
    )

    prompt = f"""Answer the question using only the context below. If the answer is not in the context, say "I don't know."

Context:
{context}

Question: {question}
"""

    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    return response["message"]["content"]


In [24]:


        # BUILD POC

def build_poc():
    print("\nExtracting PDF text...")
    pages=extract_pdf_text(PDF_PATH)
    print("Creating chunk...")
    chunks=create_chunks(pages)
    if not chunks:
        raise ValueError("no chunk wre created.")
       
        vectorstore = FAISS.from_documents(chunks, embeddings)
        vectorstore.save_local("faiss_index")  # <-- Make sure this line exists!

        print("Creating embeddings....")
        embeddings=create_embeddings(chunks)
        print("Storing vectors....")
        save_faiss_index(embeddings,chunks)
        print("\nPOC built successfully!")


In [25]:
# ASK QUESTION
def ask_questions():
    index, chunks = load_faiss_data()
    print("\nPDF RAG POC is ready")
    while True:
        question = input("\nAsk your question (type exit to stop): ").strip()
        if question.lower() == "exit":
            break
        if not question:
            print("Please enter a question.")
            continue
        
            vectorstore = FAISS.load_local(
               "faiss_index", embeddings, allow_dangerous_deserialization=True
                  )


        try:
            results = retrieve_chunks(question, index, chunks)
            if not results:
                print("No relevant information found.")
                continue

            print("\nRetrieved information:")
            for item in results:
                print(f"Pages: {item['page']} | Score: {item['score']:.4f}")

            answer = generate_answer(question, results)
            print("\nAI ANSWER:")
            print(answer)

        except ConnectionError as error:
            print("Ollama connection error:", error)
        except Exception as error:
            print("Error:", error)


In [26]:
# MAIN CODE
if __name__ == "__main__":
  try:
    build_poc()
    ask_questions()
  except Exception as error:
    print("\nAPPLICATION ERROR: ")
    print(error)




Extracting PDF text...
Creating chunk...

APPLICATION ERROR: 
name 'create_chunks' is not defined
